# Aggregating trajectories

<img align="right" src="https://movingpandas.github.io/movingpandas/assets/img/movingpandas.png">

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/movingpandas/movingpandas-examples/main?filepath=1-tutorials/9-aggregating-trajectories.ipynb)
[![IPYNB](https://img.shields.io/badge/view-ipynb-hotpink)](https://github.com/movingpandas/movingpandas-examples/blob/main/1-tutorials/9-aggregating-trajectories.ipynb)
[![HTML](https://img.shields.io/badge/view-html-green)](https://movingpandas.github.io/movingpandas-website/1-tutorials/9-aggregating-trajectories.html)

The aggregation approach implemented in TrajectoryCollectionAggregator is based on Andrienko, N., & Andrienko, G. (2011). Spatial generalization and aggregation of massive movement data. IEEE Transactions on visualization and computer graphics, 17(2), 205-219. and consists of the following main steps:

1. Extracting characteristic points from the trajectories
2. Grouping the extracted points by spatial proximity
3. Computing group centroids and corresponding Voronoi cells
4. Dividing trajectories into segments according to the Voronoi cells
5. Counting transitions from one cell to another

In [1]:
import pandas as pd
import geopandas as gpd
import movingpandas as mpd
import shapely as shp
import hvplot.pandas
import matplotlib.pyplot as plt
import folium

from geopandas import GeoDataFrame, read_file
from shapely.geometry import Point, LineString, Polygon
from datetime import datetime, timedelta
from holoviews import opts, dim

import warnings

warnings.filterwarnings("ignore")

plot_defaults = {"linewidth": 5, "capstyle": "round", "figsize": (9, 3), "legend": True}
opts.defaults(
    opts.Overlay(active_tools=["wheel_zoom"], frame_width=500, frame_height=400)
)

mpd.show_versions()

C:\Users\gfilo\AppData\Local\miniconda3\envs\geovis\Lib\site-packages\movingpandas\__init__.py:41: UserWarning: Missing optional dependencies. To use the trajectory smoother classes please install Stone Soup (see https://stonesoup.readthedocs.io/en/latest/#installation).
  warnings.warn(e.msg, UserWarning)



MovingPandas 0.22.4

SYSTEM INFO
-----------
python     : 3.11.14 | packaged by conda-forge | (main, Jan 26 2026, 23:39:55) [MSC v.1944 64 bit (AMD64)]
executable : C:\Users\gfilo\AppData\Local\miniconda3\envs\geovis\python.exe
machine    : Windows-10-10.0.26200-SP0

PROJ INFO
-----------
PROJ       : 9.7.1
PROJ data dir: C:\Users\gfilo\AppData\Local\miniconda3\envs\geovis\Library\share\proj

PYTHON DEPENDENCIES
-------------------
numpy      : 2.3.5
geopandas  : 1.1.2
geopy      : 2.4.1
geoviews   : 1.15.1
holoviews  : 1.22.1
hvplot     : 0.12.2
mapclassify: 2.10.0
matplotlib : 3.10.8
pandas     : 3.0.0
pyproj     : 3.7.2
shapely    : 2.1.2
stonesoup  : None


In [2]:
gdf = read_file("../data/geolife_small.gpkg")
tc = mpd.TrajectoryCollection(gdf, "trajectory_id", t="t")

In [3]:
tc.hvplot(line_width=7.0, tiles="CartoLight")

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Path.I     :Path   [Longitude,Latitude]
   .Path.II    :Path   [Longitude,Latitude]
   .Path.III   :Path   [Longitude,Latitude]
   .Path.IV    :Path   [Longitude,Latitude]
   .Path.V     :Path   [Longitude,Latitude]
   .Points.I   :Points   [Longitude,Latitude]   (triangle_angle)
   .Points.II  :Points   [Longitude,Latitude]   (triangle_angle)
   .Points.III :Points   [Longitude,Latitude]   (triangle_angle)
   .Points.IV  :Points   [Longitude,Latitude]   (triangle_angle)
   .Points.V   :Points   [Longitude,Latitude]   (triangle_angle)

In [4]:
tc.explore(column="trajectory_id", cmap="plasma", style_kwds={"weight": 4})

## TrajectoryCollectionAggregator

Generalizing the trip trajectories significantly speeds up the following aggregation step.

In [5]:
generalized = mpd.MinDistanceGeneralizer(tc).generalize(tolerance=100)

In [6]:
aggregator = mpd.TrajectoryCollectionAggregator(
    generalized,
    max_distance=1000,
    min_distance=100,
    min_stop_duration=timedelta(minutes=5),
)

In [7]:
pts = aggregator.get_significant_points_gdf()
clusters = aggregator.get_clusters_gdf()
(pts.hvplot(geo=True, tiles="CartoLight") * clusters.hvplot(geo=True, color="red"))

:Overlay
   .WMTS.I    :WMTS   [Longitude,Latitude]
   .Points.I  :Points   [Longitude,Latitude]
   .Points.II :Points   [Longitude,Latitude]

In [8]:
m = pts.explore(marker_kwds={"radius": 3}, name="Significant points")

clusters.explore(m=m, color="red", marker_kwds={"radius": 3}, name="Cluster centroids")

folium.TileLayer("CartoDB positron").add_to(m)
folium.LayerControl().add_to(m)

m

In [9]:
flows = aggregator.get_flows_gdf()

In [10]:
(
    flows.hvplot(
        geo=True,
        hover_cols=["weight"],
        line_width=dim("weight") * 7,
        color="#1f77b3",
        tiles="CartoLight",
    )
    * clusters.hvplot(geo=True, color="red", size=dim("n"))
)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Path.I   :Path   [Longitude,Latitude]   (weight)
   .Points.I :Points   [Longitude,Latitude]   (n)

In [11]:
m = flows.explore(
    style_kwds={"weight": 5},
    name="Flows",
)

clusters.explore(
    m=m,
    color="red",
    style_kwds={"style_function": lambda x: {"radius": x["properties"]["n"]}},
    name="Clusters",
)

folium.TileLayer("OpenStreetMap").add_to(m)
folium.LayerControl().add_to(m)

m